# Tensor-network search for two-dimensional square-QDM cage states

This notebook is intentionally separate from `cage_padding.ipynb`.  It starts from the exact constrained vertex-PEPS manifold, embeds the two-plaquette singlet as a sparse initialization, differentiates the exact finite-cluster Hamiltonian variance with Autograd, and uses quimb to optimize the shared unit tensor.

The present optimization is a **discovery calculation**, not yet a proof of a thermodynamic eigenstate.  A candidate must later pass larger-cluster transfer tests and a local telescoping/eigenstate certificate.

## Installation

Install the optional tensor-network stack with

```bash
pip install "qlinks[tn]"
```

The `tn` extra includes `quimb`, `autograd`, `numba`, and `llvmlite`.  It currently targets Python 3.11--3.13 so that Intel macOS can use the last available binary `llvmlite` wheels rather than compiling LLVM.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

import autograd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import quimb

from qlinks.caging import (
    SquareQDMChiralPEPSAnsatz,
    SquareQDMType1AdaptivePEPSFiniteClusterProblem,
    SquareQDMPEPSAnsatz,
    autograd_available,
    build_square_qdm_peps_finite_cluster_problem,
    build_square_qdm_rectangular_tile_tensor_basis,
    build_square_qdm_singlet_peps_ansatz,
    build_square_qdm_type1_adaptive_joint_cluster_problem,
    build_square_qdm_type1_adaptive_parameterization,
    build_square_qdm_type1_joint_cluster_problem,
    build_square_qdm_type1_peps_problem,
    validate_square_qdm_type1_peps_on_clusters,
    quimb_available,
    square_qdm_two_plaquette_singlet_blocks,
)
from qlinks.models import SquareQDMModel
from qlinks.visualizer import SquareQDMTensorNetworkVisualizer

print("quimb:", quimb.__version__)
print("autograd available:", autograd_available())
print("quimb available:", quimb_available())

## 1. Exact constrained unit tensor

A $3\times2$ tile owns the outgoing $+x$ and $+y$ links of its six vertices.  The virtual indices carry the boundary dimer occupations.  The structural mask therefore enforces the dimer constraint exactly when the tile is repeated in both directions.

In [ ]:
host = SquareQDMModel(
    lx=8,
    ly=8,
    boundary_condition="periodic",
    coup_kin=1.0,
    coup_pot=0.0,
)

tile_basis = build_square_qdm_rectangular_tile_tensor_basis(
    host,
    tile_shape=(3, 2),
    origin=(2, 2),
)

pd.Series({
    "tile shape": tile_basis.tile_shape,
    "owned links": tile_basis.owned_link_ids.size,
    "unconstrained configurations": 2 ** tile_basis.owned_link_ids.size,
    "physical states": tile_basis.physical_dimension,
    "allowed tensor entries": tile_basis.n_entries,
    "tensor shape (u,r,d,l,p)": tile_basis.tensor_shape,
})

In [ ]:
visualizer = SquareQDMTensorNetworkVisualizer(tile_basis)
visualizer.plot_network(n_tiles_x=3, n_tiles_y=2)
plt.show()

## 2. Structural PEPS benchmark

Setting every allowed tensor entry to one produces the equal-weight superposition of all valid dimer coverings.  On a $2\times2$ tile array, corresponding to a periodic $6\times4$ lattice, the network norm must equal the exact constrained-basis dimension.

In [ ]:
structural_ansatz = SquareQDMPEPSAnsatz(
    tile_basis=tile_basis,
    parameters=np.ones(tile_basis.n_entries, dtype=np.complex128),
)
structural_network = structural_ansatz.to_quimb_tensor_network(
    n_tiles_x=2,
    n_tiles_y=2,
)
structural_norm = float(np.real(structural_network.norm(squared=True, optimize="greedy")))

full_6x4 = SquareQDMModel(
    lx=6,
    ly=4,
    boundary_condition="periodic",
    coup_kin=1.0,
    coup_pot=0.0,
)

pd.Series({
    "PEPS norm squared": structural_norm,
    "exact dimer coverings": full_6x4.build_basis().n_states,
    "agreement": np.isclose(structural_norm, full_6x4.build_basis().n_states),
})

## 3. Embed the local two-plaquette singlet

The bare singlet occupies only two of the 108 allowed tensor entries.  Repeating it in two dimensions gives a valid dimer state, but bridge plaquettes produce a nonzero Hamiltonian variance.

In [ ]:
singlet = next(
    block
    for block in square_qdm_two_plaquette_singlet_blocks(host, directions=("x",))
    if set(block.anchor_cells) == {(2, 2), (3, 2)}
)

singlet_ansatz = build_square_qdm_singlet_peps_ansatz(
    host,
    singlet,
    origin=(2, 2),
)

nonzero_entries = np.flatnonzero(np.abs(singlet_ansatz.parameters) > 1.0e-12)
print("nonzero singlet entries:", nonzero_entries.tolist())

fig, axes = plt.subplots(1, len(nonzero_entries), figsize=(10, 4))
for axis, entry_index in zip(np.atleast_1d(axes), nonzero_entries, strict=True):
    visualizer.plot_entry(int(entry_index), ax=axis)
plt.tight_layout()
plt.show()

In [ ]:
visualizer.plot_parameter_magnitudes(
    singlet_ansatz.parameters,
    max_entries=16,
    title="Sparse two-plaquette singlet initialization",
)
plt.show()

## 4. Exact $6\times4$ optimization problem

The loss is the exact normalized energy variance

\[
\mathcal V_H(A)=
\frac{\langle\Psi(A)|(H-\langle H\rangle_A)^2|\Psi(A)\rangle}
{\langle\Psi(A)|\Psi(A)\rangle}.
\]

The finite cluster is evaluated in the $w_{00}$ sector.  The compact quimb `PTensor` exposes only the 108 structurally allowed parameters.

In [ ]:
model_6x4_w00 = SquareQDMModel(
    lx=6,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=1.0,
    coup_pot=0.0,
)

problem = build_square_qdm_peps_finite_cluster_problem(
    model_6x4_w00,
    tile_basis,
)

singlet_report = problem.diagnose(singlet_ansatz.parameters)
pd.Series({
    "Hilbert dimension": problem.hilbert_dimension,
    "nonzero amplitudes": singlet_report.nonzero_basis_amplitudes,
    "energy": singlet_report.energy.real,
    "variance": singlet_report.energy_variance,
    "residual": singlet_report.residual,
})

### Why perturb the singlet?

The exactly sparse singlet tensor is stationary inside the enlarged manifold: new entries first contribute through products around the periodic tile array.  A small reproducible perturbation activates the boundary-compatible sectors and gives Autograd a nonzero gradient.

In [ ]:
initial_parameters = problem.perturb_parameters(
    singlet_ansatz.parameters,
    scale=1.0e-2,
    seed=0,
)
initial_loss, initial_gradient = problem.loss_and_gradient_autograd(initial_parameters)
optimizer = problem.make_quimb_optimizer(initial_parameters, progbar=False)

pd.Series({
    "compact optimizer dimension": optimizer.d,
    "initial perturbed variance": initial_loss,
    "gradient norm": np.linalg.norm(initial_gradient),
})

## 5. Short quimb/Autograd demonstration

Five L-BFGS-B iterations are short enough for interactive use.  Increase `OPTIMIZATION_STEPS` for an actual search.  A decreasing loss only shows that the larger tensor manifold repairs part of the bridge leakage; zero variance and cross-size stability are required before interpreting a solution as a cage state.

In [ ]:
RUN_OPTIMIZATION = False
OPTIMIZATION_STEPS = 5

if RUN_OPTIMIZATION:
    optimization = problem.optimize_with_quimb(
        singlet_ansatz.parameters,
        max_steps=OPTIMIZATION_STEPS,
        noise_scale=1.0e-2,
        seed=0,
        autodiff_backend="autograd",
        optimizer="L-BFGS-B",
        progbar=False,
    )
    display(pd.Series({
        "initial variance": optimization.initial_loss,
        "final variance": optimization.final_loss,
        "initial residual": optimization.initial_report.residual,
        "final residual": optimization.final_report.residual,
        "function evaluations": len(optimization.loss_history),
        "improved": optimization.improved,
        "exact within tolerance": optimization.reached_exact_state,
    }))
else:
    optimization = None
    print("Set RUN_OPTIMIZATION=True to run the demonstration.")

In [ ]:
if optimization is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    visualizer.plot_optimization_history(optimization, ax=axes[0], log_scale=False)
    visualizer.plot_parameter_magnitudes(
        optimization.optimized_parameters,
        max_entries=24,
        ax=axes[1],
        title="Largest optimized unit-tensor amplitudes",
    )
    plt.tight_layout()
    plt.show()

## 6. Candidate-validation workflow

For any low-variance tensor found here:

1. rerun multiple random seeds and remove gauge/normalization redundancies;
2. validate the same unit tensor on $9\times4$, $6\times6$, and $9\times6$ clusters;
3. inspect whether the optimized support retains the cage-derived local annihilator $L_R$;
4. reconstruct simple exact coefficients where possible;
5. derive a local PEPS telescoping identity proving the eigenstate equation for arbitrary $n_x,n_y$;
6. only then combine the state certificate with the nonzero thermal $\langle Q_R\rangle$ calculation.

The visualization API is intended to make steps 1--3 inspectable: it draws the repeated tensor graph, individual boundary-resolved dimer entries, compact tensor amplitudes, and the optimization trace.

## 7. Type-1 cage structure in the PEPS objective

A type-1 cage is not searched as a generic low-variance state.  We project the PEPS amplitudes onto one kinetic-graph chiral subset and evaluate the two defining conditions separately:

\[
\mathcal L_K=\frac{\|K|\Psi_+\rangle\|^2}{N_p\langle\Psi_+|\Psi_+\rangle},
\qquad
\mathcal L_V=\frac{\operatorname{Var}_{\Psi_+}(V)}{N_p}.
\]

The first term is the destructive-interference residual on the empty chiral subset.  The second tests whether the diagonal potential is uniform on the occupied support.  The chiral projection itself is exact on the finite constrained basis.  In addition, qlinks infers a tile-periodic link-parity rule so the same symmetry can later be encoded natively in the PEPS tensor.

In [ ]:
type1_problem = build_square_qdm_type1_peps_problem(
    model_6x4_w00,
    tile_basis,
    reference_parameters=singlet_ansatz.parameters,
)

type1_report = type1_problem.diagnose(singlet_ansatz.parameters)
local_chiral_charges = type1_problem.parity_rule.tile_physical_charges(
    model_6x4_w00,
    tile_basis,
)

pd.Series({
    "selected chiral subset": type1_problem.target_chiral_label,
    "retained chiral weight": type1_report.retained_chiral_weight,
    "discarded chiral weight": type1_report.discarded_chiral_weight,
    "kinetic-interference norm": type1_report.kinetic_interference_norm,
    "kinetic-interference density": type1_report.kinetic_interference_density,
    "potential variance density": type1_report.potential_variance_density,
    "type-1 objective density": type1_report.objective,
    "nonzero opposite-sector residuals": type1_report.n_nonzero_interference_targets,
    "tile-periodic chiral rule": type1_problem.parity_rule.metadata.get("tile_periodic"),
    "C=+ local physical states": int(np.count_nonzero(local_chiral_charges == 0)),
    "C=- local physical states": int(np.count_nonzero(local_chiral_charges == 1)),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
visualizer.plot_type1_components(type1_report, ax=axes[0])
visualizer.plot_chiral_physical_charges(
    type1_problem.parity_rule,
    model_6x4_w00,
    ax=axes[1],
)
plt.tight_layout()
plt.show()

### Potential-uniformity diagnostic

For the pure kinetic model, \(V=0\) and the second condition is automatic.  Turning on the plaquette potential exposes the other type-1 requirement without changing the chiral projection.

In [ ]:
model_6x4_v1 = SquareQDMModel(
    lx=6,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=1.0,
    coup_pot=1.0,
)

type1_problem_v1 = build_square_qdm_type1_peps_problem(
    model_6x4_v1,
    tile_basis,
    reference_parameters=singlet_ansatz.parameters,
)
type1_report_v1 = type1_problem_v1.diagnose(singlet_ansatz.parameters)

pd.DataFrame(
    {
        "pure kinetic": {
            "kinetic density": type1_report.kinetic_interference_density,
            "potential mean": type1_report.potential_mean,
            "potential variance density": type1_report.potential_variance_density,
            "objective": type1_report.objective,
        },
        "K + V": {
            "kinetic density": type1_report_v1.kinetic_interference_density,
            "potential mean": type1_report_v1.potential_mean,
            "potential variance density": type1_report_v1.potential_variance_density,
            "objective": type1_report_v1.objective,
        },
    }
)

### Type-1-specific optimization

The optimizer below no longer minimizes a generic energy variance.  It varies the 108 allowed tensor entries while the finite-cluster state is kept in one chiral sector, and minimizes kinetic interference plus potential nonuniformity.  Keep the run disabled until the diagnostic cells above look sensible on your machine.

In [ ]:
type1_initial = type1_problem.base_problem.perturb_parameters(
    singlet_ansatz.parameters,
    scale=1.0e-2,
    seed=0,
)
type1_loss, type1_gradient = type1_problem.loss_and_gradient_autograd(type1_initial)

pd.Series({
    "initial type-1 objective density": type1_loss,
    "gradient norm": np.linalg.norm(type1_gradient),
})

In [ ]:
RUN_TYPE1_OPTIMIZATION = False
TYPE1_OPTIMIZATION_STEPS = 10

if RUN_TYPE1_OPTIMIZATION:
    type1_optimization = type1_problem.optimize_with_quimb(
        singlet_ansatz.parameters,
        max_steps=TYPE1_OPTIMIZATION_STEPS,
        noise_scale=1.0e-2,
        seed=0,
        autodiff_backend="autograd",
        optimizer="L-BFGS-B",
        progbar=False,
    )
    display(pd.DataFrame({
        "initial": {
            "kinetic density": type1_optimization.initial_report.kinetic_interference_density,
            "potential variance density": type1_optimization.initial_report.potential_variance_density,
            "objective": type1_optimization.initial_report.objective,
        },
        "final": {
            "kinetic density": type1_optimization.final_report.kinetic_interference_density,
            "potential variance density": type1_optimization.final_report.potential_variance_density,
            "objective": type1_optimization.final_report.objective,
        },
    }))
    visualizer.plot_type1_optimization_history(type1_optimization, log_scale=False)
    plt.show()
else:
    type1_optimization = None
    print("Set RUN_TYPE1_OPTIMIZATION=True to run the type-1-specific search.")

### Native \(\mathbb Z_2\)-symmetric tensor

The finite-cluster projector is useful for optimization, but the inferred chiral parity is also compatible with the \(3\times2\) tile itself.  qlinks therefore augments each virtual leg by one \(\mathbb Z_2\) charge bit and keeps only tensor entries satisfying local charge conservation.  Contracting a closed torus cancels all virtual charges pairwise, leaving support only in the selected global chiral subset.

This does not add variational parameters: each of the 108 structural amplitudes is copied into eight parity-compatible charge sectors.

In [ ]:
native_chiral_ansatz = SquareQDMChiralPEPSAnsatz.from_type1_problem(
    type1_problem,
    singlet_ansatz.parameters,
)

pd.Series({
    "base variational parameters": native_chiral_ansatz.n_parameters,
    "charge-augmented tensor shape": native_chiral_ansatz.tensor_shape,
    "nonzero charge-resolved entries": native_chiral_ansatz.n_nonzero_tensor_entries,
    "virtual-charge degeneracy on 2x2 torus": native_chiral_ansatz.charge_degeneracy(
        n_tiles_x=2,
        n_tiles_y=2,
    ),
})

In [ ]:
RUN_NATIVE_CHIRAL_CONTRACTION = False

if RUN_NATIVE_CHIRAL_CONTRACTION:
    native_network = native_chiral_ansatz.to_quimb_tensor_network(
        n_tiles_x=2,
        n_tiles_y=2,
    )
    native_norm_squared = native_network.norm(squared=True, optimize="greedy")
    print("native chiral PEPS norm squared:", native_norm_squared)
else:
    print(
        "Set RUN_NATIVE_CHIRAL_CONTRACTION=True to contract the charge-augmented "
        "2x2 tensor torus."
    )

## 8. Native chiral-block contraction

The projected diagnostic above is useful for checking how much of a generic PEPS lies in the selected Fock-space subset.  During the actual type-1 optimization, however, we do not need to construct amplitudes on the empty subset at all.

The native finite-cluster representation keeps only

\[
\psi_+(A)\in\mathcal H_+,
\qquad
B=K_{-,+}:\mathcal H_+\to\mathcal H_-,
\]

and evaluates the interference loss as \(\|B\psi_+(A)\|^2\).  This is the block-sparse counterpart of the charge-resolved PEPS.

In [ ]:
native_state = type1_problem.native_state_vector(
    singlet_ansatz.parameters,
)
projected_state = type1_problem.projected_state_vector(
    singlet_ansatz.parameters,
)
native_report = type1_problem.diagnose_native(
    singlet_ansatz.parameters,
)

pd.Series({
    "occupied chiral dimension": type1_problem.target_basis_indices.size,
    "empty chiral dimension": type1_problem.opposite_basis_indices.size,
    "interference block shape": type1_problem.kinetic_interference_matrix.shape,
    "native/projected state agreement": np.allclose(
        native_state,
        projected_state[type1_problem.target_basis_indices],
    ),
    "native kinetic density": native_report.kinetic_interference_density,
    "native potential density": native_report.potential_variance_density,
})

In [ ]:
native_mask = native_chiral_ansatz.native_sector_mask(
    type1_problem.base_problem,
)
native_full_state = native_chiral_ansatz.finite_cluster_state_vector(
    type1_problem.base_problem,
)

pd.Series({
    "native mask equals graph chiral subset": np.array_equal(
        native_mask,
        type1_problem.chiral_mask.astype(bool),
    ),
    "native PEPS equals projected PEPS": np.allclose(
        native_full_state,
        projected_state,
    ),
    "global Z2 charge sector": native_chiral_ansatz.global_charge_sector,
})

## 9. Joint type-1 objective across cluster sizes

A tensor that cancels interference on one short torus can overfit its periodic identifications.  We therefore optimize the same 108 amplitudes on several clusters simultaneously.

For cluster \(c\), define

\[
k_c=\frac{\|B_c\psi_{+,c}\|^2}{N_{p,c}\|\psi_{+,c}\|^2},
\qquad
v_c=\frac{\operatorname{Var}_{\psi_{+,c}}(V_c)}{N_{p,c}}.
\]

The joint loss aggregates the two mechanisms separately,

\[
\mathcal L=
\left(\sum_c w_c k_c^p\right)^{1/p}
+\lambda_V
\left(\sum_c w_c v_c^p\right)^{1/p},
\]

so a small potential error cannot hide kinetic leakage, and vice versa.

In [ ]:
model_6x2_w00 = SquareQDMModel(
    lx=6,
    ly=2,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=1.0,
    coup_pot=0.0,
)

type1_problem_6x2 = build_square_qdm_type1_peps_problem(
    model_6x2_w00,
    tile_basis,
    reference_parameters=singlet_ansatz.parameters,
)

type1_clusters = {
    "6x2": type1_problem_6x2,
    "6x4": type1_problem,
}

joint_type1_problem = build_square_qdm_type1_joint_cluster_problem(
    type1_clusters,
    aggregation_power=4.0,
    potential_weight=1.0,
)

joint_validation = validate_square_qdm_type1_peps_on_clusters(
    singlet_ansatz.parameters,
    type1_clusters,
    aggregation_power=4.0,
)

pd.DataFrame([
    {
        "cluster": record.label,
        "kinetic density": record.report.kinetic_interference_density,
        "potential density": record.report.potential_variance_density,
        "nonzero interference targets": record.report.n_nonzero_interference_targets,
    }
    for record in joint_validation.records
]).set_index("cluster")

In [ ]:
visualizer.plot_type1_cluster_validation(joint_validation)
plt.show()

pd.Series({
    "kinetic p-norm": joint_validation.kinetic_aggregate,
    "potential p-norm": joint_validation.potential_aggregate,
    "joint objective": joint_validation.objective,
    "worst cluster": joint_validation.worst_cluster_label,
})

In [ ]:
joint_initial = type1_problem_6x2.base_problem.perturb_parameters(
    singlet_ansatz.parameters,
    scale=1.0e-2,
    seed=1,
)
joint_loss, joint_gradient = joint_type1_problem.loss_and_gradient_autograd(
    joint_initial,
)

pd.Series({
    "joint Autograd loss": joint_loss,
    "diagnostic loss": joint_type1_problem.loss(joint_initial),
    "gradient norm": np.linalg.norm(joint_gradient),
})

In [ ]:
RUN_TYPE1_JOINT_OPTIMIZATION = False
TYPE1_JOINT_STEPS = 10

if RUN_TYPE1_JOINT_OPTIMIZATION:
    type1_joint_result = joint_type1_problem.optimize_with_quimb(
        singlet_ansatz.parameters,
        max_steps=TYPE1_JOINT_STEPS,
        noise_scale=1.0e-2,
        seed=1,
        progbar=True,
    )
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    visualizer.plot_type1_optimization_history(type1_joint_result, ax=axes[0])
    visualizer.plot_type1_cluster_validation(
        type1_joint_result.final_validation,
        ax=axes[1],
    )
    plt.show()
else:
    type1_joint_result = None
    print("Set RUN_TYPE1_JOINT_OPTIMIZATION=True to run the shared 6x2+6x4 search.")

### Optional longer holdout

The exact-basis map is still the scaling bottleneck, but a periodic $12\times2$ holdout remains practical on a workstation.  It is deliberately excluded from the joint training objective below.  In the first short search, reducing the $6\times2+6\times4$ objective did **not** improve this holdout, which is a useful warning that the present tensor is still learning short-torus cancellations.

In [ ]:
RUN_TYPE1_12X2_HOLDOUT = False

if RUN_TYPE1_12X2_HOLDOUT:
    model_12x2_w00 = SquareQDMModel(
        lx=12,
        ly=2,
        boundary_condition="periodic",
        winding_x=0,
        winding_y=0,
        coup_kin=1.0,
        coup_pot=0.0,
    )
    type1_problem_12x2 = build_square_qdm_type1_peps_problem(
        model_12x2_w00,
        tile_basis,
        reference_parameters=singlet_ansatz.parameters,
        infer_parity_rule=False,
    )
    holdout_parameters = (
        type1_joint_result.optimized_parameters
        if type1_joint_result is not None
        else singlet_ansatz.parameters
    )
    holdout_report = type1_problem_12x2.diagnose_native(holdout_parameters)
    pd.Series({
        "Hilbert dimension": type1_problem_12x2.base_problem.hilbert_dimension,
        "kinetic density": holdout_report.kinetic_interference_density,
        "potential density": holdout_report.potential_variance_density,
        "nonzero interference targets": holdout_report.n_nonzero_interference_targets,
    })
else:
    print("Set RUN_TYPE1_12X2_HOLDOUT=True for the longer exact-basis validation.")

## 10. Seam-resolved interference and targeted tensor enlargement

The native type-1 loss can now be decomposed before changing the ansatz.  For each plaquette,

\[
r_p = B_p\psi_+(A),
\]

qlinks records its individual norm and its coherent contribution to the total residual.  Plaquettes are grouped into tile interior, $x$ seam, $y$ seam, and four-tile corner classes.

For the repeated singlet seed, the interior components are nonzero individually but cancel exactly.  The remaining leakage is localized on tile seams.  This tells us where the one-tensor ansatz fails and which unit-cell direction should be enlarged.

In [ ]:
seam_decomposition = type1_problem.interference_decomposition(
    singlet_ansatz.parameters,
)

seam_table = pd.DataFrame([
    {
        "class": record.plaquette_class,
        "plaquettes": record.n_plaquettes,
        "after interference": record.residual_norm_squared,
        "before interference": record.incoherent_norm_squared,
        "cancellation fraction": record.cancellation_fraction,
        "nonzero targets": record.n_nonzero_targets,
    }
    for record in seam_decomposition.class_records
])

display(seam_table)
visualizer.plot_type1_interference_decomposition(seam_decomposition)
plt.show()

pd.Series({
    "total kinetic norm": seam_decomposition.total_norm_squared,
    "global cancellation fraction": seam_decomposition.global_cancellation_fraction,
    "dominant seam class": seam_decomposition.dominant_seam_class,
    "reconstruction residual": seam_decomposition.reconstruction_residual,
})

In [ ]:
sensitivity_probe = type1_problem.base_problem.perturb_parameters(
    singlet_ansatz.parameters,
    scale=1.0e-3,
    seed=0,
)

y_seam_sensitivity = type1_problem.interference_parameter_sensitivity(
    sensitivity_probe,
    "y_seam",
)

visualizer.plot_type1_parameter_sensitivity(
    y_seam_sensitivity,
    max_entries=16,
)
plt.show()

pd.DataFrame([
    {
        "entry": int(entry_index),
        "score": y_seam_sensitivity.scores[entry_index],
        "coordinate (u,r,d,l,p)": tuple(
            int(value)
            for value in tile_basis.entry_coordinates[entry_index]
        ),
        "singlet amplitude": singlet_ansatz.parameters[entry_index],
    }
    for entry_index in y_seam_sensitivity.top_entry_indices(12)
])

### Target only the offending boundary sectors

The dominant residual suggests a period-two split along $y$.  The builder duplicates only the sensitivity-ranked tensor entries; every other local amplitude remains shared.  This is a controlled two-tensor unit cell, not a position-dependent finite-cluster fit.

For cross-size tests currently accessible with exact bases, it is also useful to target the $x$ seam.  An $x$-period-two ansatz can be trained jointly on $6\times2$ and $6\times4$ and tested on the longer $12\times2$ torus.

In [ ]:
auto_adaptive = build_square_qdm_type1_adaptive_parameterization(
    type1_problem,
    singlet_ansatz.parameters,
    max_selected_entries=6,
    probe_scale=1.0e-3,
    seed=0,
)

x_adaptive = build_square_qdm_type1_adaptive_parameterization(
    type1_problem,
    singlet_ansatz.parameters,
    plaquette_class="x_seam",
    split_axis="x",
    max_selected_entries=6,
    probe_scale=1.0e-3,
    seed=0,
)

visualizer.plot_type1_adaptive_parameterization(x_adaptive)
plt.show()

pd.DataFrame([
    {
        "parameterization": "dominant-seam auto",
        "split axis": auto_adaptive.split_axis,
        "selected entries": tuple(auto_adaptive.selected_entry_indices),
        "parameter count": auto_adaptive.n_parameters,
    },
    {
        "parameterization": "longitudinal cross-size",
        "split axis": x_adaptive.split_axis,
        "selected entries": tuple(x_adaptive.selected_entry_indices),
        "parameter count": x_adaptive.n_parameters,
    },
])

In [ ]:
adaptive_joint_problem = build_square_qdm_type1_adaptive_joint_cluster_problem(
    type1_clusters,
    x_adaptive,
    aggregation_power=4.0,
)

adaptive_initial = x_adaptive.lift_parameters(
    singlet_ansatz.parameters,
)
adaptive_initial_validation = adaptive_joint_problem.diagnose(
    adaptive_initial,
)

visualizer.plot_type1_cluster_validation(adaptive_initial_validation)
plt.show()

pd.Series({
    "base parameters": tile_basis.n_entries,
    "adaptive parameters": x_adaptive.n_parameters,
    "initial adaptive objective": adaptive_initial_validation.objective,
    "worst training cluster": adaptive_initial_validation.worst_cluster_label,
})

In [ ]:
RUN_ADAPTIVE_SCIPY_OPTIMIZATION = False
RUN_ADAPTIVE_QUIMB_OPTIMIZATION = False
ADAPTIVE_STEPS = 10

if RUN_ADAPTIVE_SCIPY_OPTIMIZATION:
    adaptive_result = adaptive_joint_problem.optimize_with_scipy(
        adaptive_initial,
        max_steps=ADAPTIVE_STEPS,
        noise_scale=1.0e-3,
        seed=0,
    )
elif RUN_ADAPTIVE_QUIMB_OPTIMIZATION:
    adaptive_result = adaptive_joint_problem.optimize_with_quimb(
        adaptive_initial,
        max_steps=ADAPTIVE_STEPS,
        noise_scale=1.0e-3,
        seed=0,
        progbar=True,
    )
else:
    adaptive_result = None
    print(
        "Enable one adaptive optimization flag to search the targeted "
        "period-two unit cell."
    )

if adaptive_result is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    visualizer.plot_type1_optimization_history(adaptive_result, ax=axes[0])
    visualizer.plot_type1_cluster_validation(
        adaptive_result.final_validation,
        ax=axes[1],
    )
    plt.show()

### Optional $12\times2$ adaptive holdout

The holdout is deliberately excluded from training.  In a short analytic-gradient benchmark, the targeted $x$-period-two enlargement reduced the shared $6\times2+6\times4$ objective and also reduced the $12\times2$ kinetic-interference density.  This is the first indication that a selectively enlarged unit cell transfers better than the original one-tensor optimization, but it is still far from a zero-residual type-1 cage.

With the deterministic six-entry $x$-seam selection
``[8, 9, 21, 24, 58, 60]`` and eight analytic-gradient steps, the current
benchmark changes

\[
\mathcal L_{6\times2,6\times4}: 0.180606 \rightarrow 0.138872,
\]

while the untrained holdout changes

\[
k_{12\times2}: 0.208334 \rightarrow 0.162702.
\]

This is transfer along the longitudinal direction, not yet the true two-dimensional result: the larger transverse $y$-seam residual still needs training and validation on clusters with more tensor rows.

In [ ]:
RUN_ADAPTIVE_12X2_HOLDOUT = False

if RUN_ADAPTIVE_12X2_HOLDOUT:
    if "type1_problem_12x2" not in globals():
        model_12x2_w00 = SquareQDMModel(
            lx=12,
            ly=2,
            boundary_condition="periodic",
            winding_x=0,
            winding_y=0,
            coup_kin=1.0,
            coup_pot=0.0,
        )
        type1_problem_12x2 = build_square_qdm_type1_peps_problem(
            model_12x2_w00,
            tile_basis,
            reference_parameters=singlet_ansatz.parameters,
            infer_parity_rule=False,
        )

    adaptive_holdout = SquareQDMType1AdaptivePEPSFiniteClusterProblem.from_problem(
        type1_problem_12x2,
        x_adaptive,
    )
    holdout_parameters = (
        adaptive_result.optimized_parameters
        if adaptive_result is not None
        else adaptive_initial
    )
    display(pd.Series({
        "initial kinetic density": adaptive_holdout.diagnose(
            adaptive_initial,
        ).kinetic_interference_density,
        "selected kinetic density": adaptive_holdout.diagnose(
            holdout_parameters,
        ).kinetic_interference_density,
        "Hilbert dimension": type1_problem_12x2.base_problem.hilbert_dimension,
    }))
else:
    print("Set RUN_ADAPTIVE_12X2_HOLDOUT=True to build the longer exact holdout.")

## 11. Next structural step

The finite-basis calculation is now symmetry adapted: it constructs only the occupied chiral block and the rectangular interference map.  The next scaling step is to express the same quantities as local PEPS contractions, so that \(B\psi_+\), the potential moments, and their gradients can be evaluated without enumerating the full dimer basis.

A convincing candidate must make both the kinetic-interference density and the potential-variance density decrease on every added two-dimensional cluster.  Only after that should its tensor entries be reconstructed into an exact local telescoping identity.